In [29]:
import pandas as pd
import numpy as np
training_dataset = pd.read_csv(
    "../data/processed/training_dataset.csv"
)


FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/training_dataset.csv'

In [2]:
print("Available variables:")

for name in [
    "training_dataset",
    "training",
    "X_train",
    "y_train",
    "X_val",
    "y_val",
    "X_test",
    "y_test"
]:
    print(name, "->", name in globals())

Available variables:
training_dataset -> False
training -> False
X_train -> False
y_train -> False
X_val -> False
y_val -> False
X_test -> False
y_test -> False


In [3]:
from pathlib import Path

data_path = Path("../data/processed")

print("Processed files:")
for file in sorted(data_path.iterdir()):
    print(file.name)

Processed files:
enhanced_feature_dataset.csv
external_feature_dataset.csv
feature_dataset.csv
model_a_features.csv
sepsis_progression_test.csv
sepsis_progression_train.csv
sepsis_progression_validation.csv


In [4]:
import pandas as pd

train = pd.read_csv(
    "../data/processed/sepsis_progression_train.csv"
)

validation = pd.read_csv(
    "../data/processed/sepsis_progression_validation.csv"
)

test = pd.read_csv(
    "../data/processed/sepsis_progression_test.csv"
)

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

print("\nTrain columns:")
print(train.columns.tolist())

Train shape: (300569, 58)
Validation shape: (64153, 58)
Test shape: (64226, 58)

Train columns:
['hour', 'heart_rate', 'resp_rate', 'temperature', 'sbp', 'dbp', 'mbp', 'spo2', 'gcs', 'creatinine', 'bun', 'urineoutput_last', 'urineoutput_sum', 'urineoutput_24hr', 'wbc', 'hemoglobin', 'hematocrit', 'platelet', 'bands', 'sodium', 'potassium', 'chloride', 'bicarbonate', 'calcium', 'magnesium', 'aniongap', 'albumin', 'bilirubin_total', 'bilirubin_max', 'inr', 'pt', 'ptt', 'crp', 'lactate', 'pao2fio2ratio_novent', 'pao2fio2ratio_vent', 'glucose_lab', 'shock_index', 'map_calculated', 'bun_creatinine_ratio', 'spo2_deficit', 'heart_rate_prev', 'heart_rate_delta', 'resp_rate_prev', 'resp_rate_delta', 'temperature_prev', 'temperature_delta', 'sbp_prev', 'sbp_delta', 'mbp_prev', 'mbp_delta', 'spo2_prev', 'spo2_delta', 'gcs_prev', 'gcs_delta', 'heart_rate_roll3_mean', 'heart_rate_roll6_mean', 'progression_class']


In [5]:
# Create binary deterioration target

train["deterioration"] = (
    train["progression_class"] == 2
).astype(int)

validation["deterioration"] = (
    validation["progression_class"] == 2
).astype(int)

test["deterioration"] = (
    test["progression_class"] == 2
).astype(int)


# Check distributions

print("TRAIN")
print(train["deterioration"].value_counts().sort_index())
print(train["deterioration"].value_counts(normalize=True).sort_index() * 100)

print("\nVALIDATION")
print(validation["deterioration"].value_counts().sort_index())
print(validation["deterioration"].value_counts(normalize=True).sort_index() * 100)

print("\nTEST")
print(test["deterioration"].value_counts().sort_index())
print(test["deterioration"].value_counts(normalize=True).sort_index() * 100)

TRAIN
deterioration
0    231522
1     69047
Name: count, dtype: int64
deterioration
0    77.027904
1    22.972096
Name: proportion, dtype: float64

VALIDATION
deterioration
0    49288
1    14865
Name: count, dtype: int64
deterioration
0    76.828831
1    23.171169
Name: proportion, dtype: float64

TEST
deterioration
0    49207
1    15019
Name: count, dtype: int64
deterioration
0    76.615389
1    23.384611
Name: proportion, dtype: float64


In [6]:
# Target column
target_column = "deterioration"

# Columns that must NOT be used as model features
excluded_columns = [
    "progression_class",
    "deterioration"
]

# Create feature list
feature_columns = [
    col for col in train.columns
    if col not in excluded_columns
]

X_train = train[feature_columns].copy()
y_train = train[target_column].copy()

X_val = validation[feature_columns].copy()
y_val = validation[target_column].copy()

X_test = test[feature_columns].copy()
y_test = test[target_column].copy()

print("Number of features:", len(feature_columns))

print("\nTraining:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nValidation:")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nTest:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nExcluded columns:")
print(excluded_columns)

print("\nFeatures:")
print(feature_columns)

Number of features: 57

Training:
X_train: (300569, 57)
y_train: (300569,)

Validation:
X_val: (64153, 57)
y_val: (64153,)

Test:
X_test: (64226, 57)
y_test: (64226,)

Excluded columns:
['progression_class', 'deterioration']

Features:
['hour', 'heart_rate', 'resp_rate', 'temperature', 'sbp', 'dbp', 'mbp', 'spo2', 'gcs', 'creatinine', 'bun', 'urineoutput_last', 'urineoutput_sum', 'urineoutput_24hr', 'wbc', 'hemoglobin', 'hematocrit', 'platelet', 'bands', 'sodium', 'potassium', 'chloride', 'bicarbonate', 'calcium', 'magnesium', 'aniongap', 'albumin', 'bilirubin_total', 'bilirubin_max', 'inr', 'pt', 'ptt', 'crp', 'lactate', 'pao2fio2ratio_novent', 'pao2fio2ratio_vent', 'glucose_lab', 'shock_index', 'map_calculated', 'bun_creatinine_ratio', 'spo2_deficit', 'heart_rate_prev', 'heart_rate_delta', 'resp_rate_prev', 'resp_rate_delta', 'temperature_prev', 'temperature_delta', 'sbp_prev', 'sbp_delta', 'mbp_prev', 'mbp_delta', 'spo2_prev', 'spo2_delta', 'gcs_prev', 'gcs_delta', 'heart_rate_roll3

In [7]:
missing_summary = pd.DataFrame({
    "missing_count": X_train.isna().sum(),
    "missing_percent": X_train.isna().mean() * 100
})

print("Features with <50% missing:")
print(
    missing_summary[
        missing_summary["missing_percent"] < 50
    ].sort_values("missing_percent")
)

print("\nFeatures with >=50% missing:")
print(
    missing_summary[
        missing_summary["missing_percent"] >= 50
    ].sort_values("missing_percent")
)

Features with <50% missing:
                       missing_count  missing_percent
hour                               0         0.000000
heart_rate_roll6_mean             67         0.022291
heart_rate_roll3_mean            426         0.141731
heart_rate                      3193         1.062318
mbp                             4512         1.501153
sbp                             5093         1.694453
dbp                             5133         1.707761
map_calculated                  5160         1.716744
resp_rate                       5191         1.727058
heart_rate_prev                 5355         1.781621
shock_index                     5952         1.980244
resp_rate_prev                  7241         2.409097
spo2_deficit                    7350         2.445362
spo2                            7350         2.445362
heart_rate_delta                7587         2.524212
mbp_prev                        9277         3.086479
spo2_prev                       9493         3.158343


In [8]:
completely_missing = [
    col for col in X_train.columns
    if X_train[col].isna().all()
]

print("Completely missing features:", completely_missing)

Completely missing features: []


In [9]:
missing_by_class = pd.DataFrame({
    "feature": X_train.columns,
    "missing_not_deteriorating": [
        X_train.loc[y_train == 0, col].isna().mean() * 100
        for col in X_train.columns
    ],
    "missing_deteriorating": [
        X_train.loc[y_train == 1, col].isna().mean() * 100
        for col in X_train.columns
    ]
})

missing_by_class["difference"] = (
    missing_by_class["missing_deteriorating"]
    - missing_by_class["missing_not_deteriorating"]
)

print(
    missing_by_class
    .sort_values("difference", key=abs, ascending=False)
    .head(20)
)

                 feature  missing_not_deteriorating  missing_deteriorating  \
9             creatinine                  89.211392              96.088172   
10                   bun                  89.244219              96.093965   
39  bun_creatinine_ratio                  89.261496              96.109896   
21              chloride                  88.663712              95.459614   
20             potassium                  88.370868              95.165612   
22           bicarbonate                  89.285683              96.069344   
19                sodium                  88.612313              95.364027   
25              aniongap                  89.336651              96.088172   
36           glucose_lab                  89.401439              96.054861   
24             magnesium                  89.712857              96.053413   
17              platelet                  89.840274              96.004171   
16            hematocrit                  88.071976             

In [10]:
import lightgbm as lgb
print("lightgbm version :", lgb.__version__)
binary_model = lgb.LGBMClassifier(
    objective = "binary",
    n_estimators = 300, ### no of binary tree
    learning_rate = 0.05,
    random_state = 42,
    n_jobs = -1,
)
binary_model.fit(
    X_train,
    y_train
)
print("Binary baseline model trained successfully!")

lightgbm version : 4.6.0
[LightGBM] [Info] Number of positive: 69047, number of negative: 231522
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.042902 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 9089
[LightGBM] [Info] Number of data points in the train set: 300569, number of used features: 57
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.229721 -> initscore=-1.209887
[LightGBM] [Info] Start training from score -1.209887
Binary baseline model trained successfully!


In [12]:
#probability of deterioration
y_val_proba = binary_model.predict_proba(X_val)[:,1]
print("Validation probabilities generated successfully!")
print("shape:", y_val_proba.shape)
print("\nfirst 20 probability:")
print(y_val_proba[:20])
print("\nProbability range:")
print("Minimum:", y_val_proba.min())
print("Maximum:", y_val_proba.max())
print("Mean:", y_val_proba.mean())

Validation probabilities generated successfully!
shape: (64153,)

first 20 probability:
[0.166976   0.07202859 0.20140725 0.13235384 0.12363743 0.43078575
 0.16648364 0.40957513 0.32740573 0.31438602 0.22945069 0.25851806
 0.24029391 0.03573347 0.49298146 0.33834824 0.21001978 0.21439636
 0.29023638 0.26398663]

Probability range:
Minimum: 0.00641455087264449
Maximum: 0.770031596943584
Mean: 0.22990593485253408


In [13]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

# Default threshold
threshold = 0.50

y_val_pred = (
    y_val_proba >= threshold
).astype(int)  ### astype is a pandas method used for the conversion of value eg astype(int) conversion of value to int

print("Validation predictions generated successfully!")

print("\nPredicted distribution:")
print(
    pd.Series(y_val_pred)
    .value_counts()
    .sort_index() ## sort_index is a pandas method tht sorts the datafram based on the index
)

print("\nActual distribution:")
print(
    y_val.value_counts()
    .sort_index()
)

print("\nAccuracy:")
print(
    accuracy_score(y_val, y_val_pred)
)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred,
        target_names=[
            "Not deteriorating",
            "Deteriorating"
        ]
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(y_val, y_val_pred)
)

print("\nROC-AUC:")
print(
    roc_auc_score(y_val, y_val_proba)
)

print("\nPR-AUC:")
print(
    average_precision_score(y_val, y_val_proba)
)

Validation predictions generated successfully!

Predicted distribution:
0    63571
1      582
Name: count, dtype: int64

Actual distribution:
deterioration
0    49288
1    14865
Name: count, dtype: int64

Accuracy:
0.7692235748912756

Classification Report:
                   precision    recall  f1-score   support

Not deteriorating       0.77      0.99      0.87     49288
    Deteriorating       0.55      0.02      0.04     14865

         accuracy                           0.77     64153
        macro avg       0.66      0.51      0.46     64153
     weighted avg       0.72      0.77      0.68     64153


Confusion Matrix:
[[49027   261]
 [14544   321]]

ROC-AUC:
0.6961630060087943

PR-AUC:
0.3846137251751667


In [14]:
print("TRAIN TARGET CHECK")
print(
    train.groupby("deterioration")["progression_class"]
    .value_counts()
)

print("\nBinary target counts:")
print(train["deterioration"].value_counts().sort_index())

TRAIN TARGET CHECK
deterioration  progression_class
0              1                    146483
               0                     85039
1              2                     69047
Name: count, dtype: int64

Binary target counts:
deterioration
0    231522
1     69047
Name: count, dtype: int64


In [15]:
progression_train = pd.read_csv(
    "../data/processed/sepsis_progression_train.csv"
)

print(progression_train.columns.tolist())

['hour', 'heart_rate', 'resp_rate', 'temperature', 'sbp', 'dbp', 'mbp', 'spo2', 'gcs', 'creatinine', 'bun', 'urineoutput_last', 'urineoutput_sum', 'urineoutput_24hr', 'wbc', 'hemoglobin', 'hematocrit', 'platelet', 'bands', 'sodium', 'potassium', 'chloride', 'bicarbonate', 'calcium', 'magnesium', 'aniongap', 'albumin', 'bilirubin_total', 'bilirubin_max', 'inr', 'pt', 'ptt', 'crp', 'lactate', 'pao2fio2ratio_novent', 'pao2fio2ratio_vent', 'glucose_lab', 'shock_index', 'map_calculated', 'bun_creatinine_ratio', 'spo2_deficit', 'heart_rate_prev', 'heart_rate_delta', 'resp_rate_prev', 'resp_rate_delta', 'temperature_prev', 'temperature_delta', 'sbp_prev', 'sbp_delta', 'mbp_prev', 'mbp_delta', 'spo2_prev', 'spo2_delta', 'gcs_prev', 'gcs_delta', 'heart_rate_roll3_mean', 'heart_rate_roll6_mean', 'progression_class']


In [16]:
# Inspect the original hourly SOFA dataset

sofa_hourly = pd.read_csv(
    "../data/hf_dataset/sepsis_sofa_hourly_clock.csv"
)

print("SOFA hourly shape:")
print(sofa_hourly.shape)

print("\nSOFA hourly columns:")
print(sofa_hourly.columns.tolist())

print("\nFirst 5 rows:")
print(sofa_hourly.head())

SOFA hourly shape:
(812621, 29)

SOFA hourly columns:
['stay_id', 'hr', 'starttime', 'endtime', 'pao2fio2ratio_novent', 'pao2fio2ratio_vent', 'rate_epinephrine', 'rate_norepinephrine', 'rate_dopamine', 'rate_dobutamine', 'meanbp_min', 'gcs_min', 'uo_24hr', 'bilirubin_max', 'creatinine_max', 'platelet_min', 'respiration', 'coagulation', 'liver', 'cardiovascular', 'cns', 'renal', 'respiration_24hours', 'coagulation_24hours', 'liver_24hours', 'cardiovascular_24hours', 'cns_24hours', 'renal_24hours', 'sofa_24hours']

First 5 rows:
    stay_id  hr            starttime              endtime  \
0  30000484   0  2136-01-14 18:00:00  2136-01-14 19:00:00   
1  30000484   1  2136-01-14 19:00:00  2136-01-14 20:00:00   
2  30000484   2  2136-01-14 20:00:00  2136-01-14 21:00:00   
3  30000484   3  2136-01-14 21:00:00  2136-01-14 22:00:00   
4  30000484   4  2136-01-14 22:00:00  2136-01-14 23:00:00   

   pao2fio2ratio_novent  pao2fio2ratio_vent  rate_epinephrine  \
0                   NaN            

In [17]:
# Check whether the original feature dataset contains patient/time identifiers

feature_df = pd.read_csv(
    "../data/processed/feature_dataset.csv"
)

print("Feature dataset shape:", feature_df.shape)
print("\nColumns:")
print(feature_df.columns.tolist())

print("\nPossible identifier columns:")
possible_ids = [
    col for col in feature_df.columns
    if any(key in col.lower() for key in [
        "stay", "patient", "subject", "hadm", "hour", "time"
    ])
]

print(possible_ids)

Feature dataset shape: (20336, 201)

Columns:
['HR_mean', 'HR_min', 'HR_max', 'HR_std', 'HR_last', 'O2Sat_mean', 'O2Sat_min', 'O2Sat_max', 'O2Sat_std', 'O2Sat_last', 'Temp_mean', 'Temp_min', 'Temp_max', 'Temp_std', 'Temp_last', 'SBP_mean', 'SBP_min', 'SBP_max', 'SBP_std', 'SBP_last', 'MAP_mean', 'MAP_min', 'MAP_max', 'MAP_std', 'MAP_last', 'DBP_mean', 'DBP_min', 'DBP_max', 'DBP_std', 'DBP_last', 'Resp_mean', 'Resp_min', 'Resp_max', 'Resp_std', 'Resp_last', 'EtCO2_mean', 'EtCO2_min', 'EtCO2_max', 'EtCO2_std', 'EtCO2_last', 'BaseExcess_mean', 'BaseExcess_min', 'BaseExcess_max', 'BaseExcess_std', 'BaseExcess_last', 'HCO3_mean', 'HCO3_min', 'HCO3_max', 'HCO3_std', 'HCO3_last', 'FiO2_mean', 'FiO2_min', 'FiO2_max', 'FiO2_std', 'FiO2_last', 'pH_mean', 'pH_min', 'pH_max', 'pH_std', 'pH_last', 'PaCO2_mean', 'PaCO2_min', 'PaCO2_max', 'PaCO2_std', 'PaCO2_last', 'SaO2_mean', 'SaO2_min', 'SaO2_max', 'SaO2_std', 'SaO2_last', 'AST_mean', 'AST_min', 'AST_max', 'AST_std', 'AST_last', 'BUN_mean', 'BUN_m

In [18]:
import pandas as pd

feature_dataset = pd.read_csv(
    "../data/processed/feature_dataset.csv"
)

print("Feature dataset loaded successfully!")
print("Shape:", feature_dataset.shape)

Feature dataset loaded successfully!
Shape: (20336, 201)


In [20]:
# Inspect the structure and target of the feature dataset

print("Shape:", feature_dataset.shape)

print("\nTarget distribution:")
print(feature_dataset["SepsisLabel"].value_counts())

print("\nTarget proportions:")
print(feature_dataset["SepsisLabel"].value_counts(normalize=True))

print("\nData types:")
print(feature_dataset.dtypes.value_counts())

print("\nMissing values - top 20:")
print(
    feature_dataset.isnull()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

print("\nRows with target missing:")
print(feature_dataset["SepsisLabel"].isna().sum())

print("\nUnique target values:")
print(feature_dataset["SepsisLabel"].unique())

Shape: (20336, 201)

Target distribution:
SepsisLabel
0    18546
1     1790
Name: count, dtype: int64

Target proportions:
SepsisLabel
0    0.911979
1    0.088021
Name: proportion, dtype: float64

Data types:
float64    194
int64        7
Name: count, dtype: int64

Missing values - top 20:
EtCO2_last               20336
EtCO2_min                20336
EtCO2_std                20336
EtCO2_mean               20336
EtCO2_max                20336
TroponinI_last           20326
Bilirubin_direct_last    20320
Fibrinogen_last          20268
Bilirubin_total_last     20186
Alkalinephos_last        20163
AST_last                 20160
TroponinI_std            20065
Bilirubin_direct_std     20060
Lactate_last             20052
SaO2_last                19900
TroponinI_max            19847
TroponinI_min            19847
TroponinI_mean           19847
PTT_last                 19764
Bilirubin_direct_mean    19750
dtype: int64

Rows with target missing:
0

Unique target values:
[0 1]


In [21]:
print("First 10 rows:")
display(feature_dataset.head(10))

print("\nTarget column:")
print(feature_dataset["SepsisLabel"].head(20).to_list())

print("\nLast columns:")
print(feature_dataset.columns[-10:].tolist())

First 10 rows:


,HR_mean,HR_min,HR_max,HR_std,HR_last,O2Sat_mean,O2Sat_min,O2Sat_max,O2Sat_std,O2Sat_last,...,HospAdmTime_min,HospAdmTime_max,HospAdmTime_std,HospAdmTime_last,ICULOS_mean,ICULOS_min,ICULOS_max,ICULOS_std,ICULOS_last,SepsisLabel
0,101.571429,76.0,117.0,9.594378,84.0,91.477273,85.0,100.0,3.460667,85.0,...,-0.03,-0.03,3.502025e-18,-0.03,27.5,1,54,15.732133,54,0
1,60.954545,54.0,94.0,8.144395,55.0,97.000000,94.0,100.0,2.138090,95.0,...,-98.60,-98.60,2.906048e-14,-98.60,12.0,1,23,6.782330,23,0
2,79.611111,68.0,93.0,6.714036,78.0,95.431818,91.0,99.0,1.655122,97.0,...,-1195.71,-1195.71,0.000000e+00,-1195.71,24.5,1,48,14.000000,48,0
3,102.444444,93.0,113.0,6.337212,NaN,98.203704,95.5,100.0,1.449531,NaN,...,-8.77,-8.77,1.807799e-15,-8.77,15.0,1,29,8.514693,29,0
4,73.916667,61.0,88.0,7.586697,NaN,97.500000,96.0,99.0,0.741620,NaN,...,-0.05,-0.05,7.012323e-18,-0.05,25.5,2,49,14.000000,49,0
5,100.000000,87.0,111.0,7.402702,110.0,98.437500,95.0,100.0,1.263263,98.0,...,-0.03,-0.03,0.000000e+00,-0.03,11.0,3,19,5.049752,19,0
6,120.363636,103.0,155.5,10.980090,103.0,95.409091,93.0,100.0,1.295218,96.5,...,-0.05,-0.05,2.105191e-17,-0.05,23.0,1,45,13.133926,45,0
7,76.342105,65.0,88.0,6.226692,84.0,97.805556,79.0,100.0,4.254876,95.0,...,-2.23,-2.23,0.000000e+00,-2.23,20.5,1,40,11.690452,40,0
8,112.647059,85.0,143.0,12.857603,NaN,97.930672,89.5,100.0,1.917173,NaN,...,-0.03,-0.03,1.042857e-17,-0.03,129.5,1,258,74.622383,258,1
9,77.043478,63.0,84.0,5.708672,79.0,95.826087,90.0,100.0,2.757670,92.0,...,-2.36,-2.36,0.000000e+00,-2.36,14.0,3,25,6.782330,25,0



Target column:
[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0]

Last columns:
['HospAdmTime_min', 'HospAdmTime_max', 'HospAdmTime_std', 'HospAdmTime_last', 'ICULOS_mean', 'ICULOS_min', 'ICULOS_max', 'ICULOS_std', 'ICULOS_last', 'SepsisLabel']


In [22]:
print("Index information:")
print(feature_dataset.index[:10])

print("\nICULOS statistics:")
print(feature_dataset["ICULOS_last"].describe())

print("\nSepsisLabel by ICU time:")
print(
    feature_dataset.groupby("SepsisLabel")["ICULOS_last"]
    .describe()
)

print("\nHospAdmTime statistics:")
print(feature_dataset["HospAdmTime_last"].describe())

Index information:
RangeIndex(start=0, stop=10, step=1)

ICULOS statistics:
count    20336.000000
mean        39.774194
std         22.552482
min          8.000000
25%         26.000000
50%         40.000000
75%         48.000000
max        336.000000
Name: ICULOS_last, dtype: float64

SepsisLabel by ICU time:
               count       mean        std  min   25%   50%   75%    max
SepsisLabel                                                             
0            18546.0  37.866009  13.919869  8.0  26.0  40.0  47.0  336.0
1             1790.0  59.544693  57.826063  8.0  17.0  38.0  81.0  336.0

HospAdmTime statistics:
count    20335.000000
mean       -48.678411
std        143.683318
min      -3710.660000
25%        -34.135000
50%         -2.770000
75%         -0.020000
max         23.990000
Name: HospAdmTime_last, dtype: float64


In [23]:
import pandas as pd

train_df = pd.read_csv(
    "../data/processed/sepsis_progression_train.csv"
)

val_df = pd.read_csv(
    "../data/processed/sepsis_progression_validation.csv"
)

test_df = pd.read_csv(
    "../data/processed/sepsis_progression_test.csv"
)

print("Datasets loaded successfully!")

print("\nTrain shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Datasets loaded successfully!

Train shape: (300569, 58)
Validation shape: (64153, 58)
Test shape: (64226, 58)


In [24]:
###Audit the actual progression target

print("Progression class distribution:")
print(train_df["progression_class"].value_counts().sort_index())

print("\nProgression class proportions:")
print(
    train_df["progression_class"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nUnique progression classes:")
print(sorted(train_df["progression_class"].unique()))

Progression class distribution:
progression_class
0     85039
1    146483
2     69047
Name: count, dtype: int64

Progression class proportions:
progression_class
0    0.282927
1    0.487352
2    0.229721
Name: proportion, dtype: float64

Unique progression classes:
[np.int64(0), np.int64(1), np.int64(2)]


In [25]:
# Search for SOFA-related information


sofa_columns = [
    col for col in train_df.columns
    if "sofa" in col.lower()
]

print("SOFA-related columns:")
print(sofa_columns)

print("\nNumber of SOFA-related columns:", len(sofa_columns))

SOFA-related columns:
[]

Number of SOFA-related columns: 0


In [28]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

thresholds = np.arange(0.05, 0.51, 0.01)

results = []

for threshold in thresholds:
    y_pred = (val_prob >= threshold).astype(int)

    report = classification_report(
        y_val,
        y_pred,
        output_dict=True,
        zero_division=0
    )

    results.append({
        "threshold": threshold,
        "precision": report["1"]["precision"],
        "recall": report["1"]["recall"],
        "f1": report["1"]["f1-score"]
    })

results_df = pd.DataFrame(results)

print(results_df.sort_values("f1", ascending=False).head(10))

NameError: name 'val_prob' is not defined